# Automatic Differentiation

## ✅ Exercise 41 — Gradient of a scalar function

In [1]:
import jax
import jax.numpy as jnp

f = lambda x: x ** 2
grad_f = jax.grad(f)
grad_f(3.0)  # → 6.0

Array(6., dtype=float32, weak_type=True)

## ✅ Exercise 42 — Gradient of a multi-term polynomial

In [2]:
f = lambda x: 3*x**3 - 2*x**2 + x - 5
grad_f = jax.grad(f)
grad_f(2.0)  # f'(x) = 9x² - 4x + 1 → 9(4) - 4(2) + 1 = 29.0

Array(29., dtype=float32, weak_type=True)

## ✅ Exercise 43 — value_and_grad

In [4]:
f = lambda x: jnp.sin(x)
val, grad = jax.value_and_grad(f)(jnp.pi / 4)
# val  → sin(π/4) ≈ 0.7071
# grad → cos(π/4) ≈ 0.7071
val, grad

(Array(0.70710677, dtype=float32, weak_type=True),
 Array(0.70710677, dtype=float32, weak_type=True))

## ✅ Exercise 44 — Gradient w.r.t. specific argument

In [7]:
f = lambda x, y: x**2 + y**3
grad_wrt_y = jax.grad(f, argnums=1)
grad_wrt_y(2.0, 3.0)  # f'_y = 3y² → 27.0

Array(27., dtype=float32, weak_type=True)

## ✅ Exercise 45 — Gradient over a vector

In [14]:
w = jnp.array([1.0, 2.0, 3.0])
jax.grad(lambda w: jnp.dot(w, w))(w)  # → 2w = [2., 4., 6.]

Array([2., 4., 6.], dtype=float32)

## ✅ Exercise 46 — Second derivative

In [8]:
f        = lambda x: x ** 4
df       = jax.grad(f)
d2f      = jax.grad(df)
d2f(2.0)  # f''(x) = 12x² → 48.0

Array(48., dtype=float32, weak_type=True)

## ✅ Exercise 47 — JIT + grad

In [9]:
import time

f        = lambda x: jnp.tanh(x)
grad_f   = jax.jit(jax.grad(f))

grad_f(1.0)  # warm up

def benchmark(fn, x, runs=1000):
    start = time.perf_counter()
    for _ in range(runs):
        jax.block_until_ready(fn(x))
    return (time.perf_counter() - start) / runs

x = jnp.array(1.0)
# benchmark shows JIT grad significantly faster than eager grad
print(f"JIT grad: {benchmark(grad_f, x) * 1e6:.2f} µs")
print(f"Eager   : {benchmark(jax.grad(f), x) * 1e6:.2f} µs")

W0304 01:42:07.799224 10164927 cpp_gen_intrinsics.cc:74] Empty bitcode string provided for eigen. Optimizations relying on this IR will be disabled.


JIT grad: 7.45 µs
Eager   : 467.83 µs


## ✅ Exercise 48 — Jacobian of a vector function

`grad` only works scalar outputs.
When your function outputs a **vector**, you need the **Jacobian matrix** — all partial derivatives of all outputs w.r.t. all inputs.

In [10]:
def f(x):
    return jnp.array([x[0]**2, x[0]*x[1], x[1]**3])

x = jnp.array([2.0, 3.0])
J = jax.jacobian(f)(x)
# J[i, j] = ∂f_i / ∂x_j
# → [[4., 0.],
#    [3., 2.],
#    [0., 27.]]
print(J)

[[ 4.  0.]
 [ 3.  2.]
 [ 0. 27.]]


## ✅ Exercise 49 — grad with reduction over matrix

Works identically to vector case — JAX treats the matrix as a flat parameter space.
This is how weight matrices in neural networks receive gradients.

In [11]:
W = jnp.ones((3, 3))
f = lambda W: jnp.sum(W ** 2)
jax.grad(f)(W)
# f = Σ wᵢⱼ²  →  ∂f/∂wᵢⱼ = 2wᵢⱼ
# All ones → gradient is all 2.0s → (3,3) matrix of 2.0

Array([[2., 2., 2.],
       [2., 2., 2.],
       [2., 2., 2.]], dtype=float32)

## ✅ Exercise 50 — Stop gradient

Sometimes you want to freeze part of a computation — treat it as a constant during differentiation.

In [12]:
# f(x) = x * stop_gradient(x + 1)
# Treating (x+1) as constant c:
# f(x) = c * x  →  f'(x) = c = (x+1)
# Without stop_gradient: f'(x) = 2x + 1

f_stopped = lambda x: x * jax.lax.stop_gradient(x + 1)
f_normal  = lambda x: x * (x + 1)

x = 3.0
print(jax.grad(f_stopped)(x))  # → x+1 = 4.0  (frozen term ignored)
print(jax.grad(f_normal)(x))   # → 2x+1 = 7.0

4.0
7.0
